In [1]:
import requests
from urllib.parse import quote

query = quote("artificial intelligence")
api_key = "CUSTOM_API_KEY"
cx="SEARCH_ENGINE_KEY"
link_site = "reddit.com"
google_search_url = f"https://www.googleapis.com/customsearch/v1?key={api_key}&cx={cx}&q={query}"
r = requests.get(google_search_url)
r.headers, query

({'Vary': 'Origin, X-Origin, Referer', 'Content-Type': 'application/json; charset=UTF-8', 'Content-Encoding': 'gzip', 'Date': 'Tue, 16 Sep 2025 14:09:12 GMT', 'Server': 'ESF', 'X-XSS-Protection': '0', 'X-Frame-Options': 'SAMEORIGIN', 'X-Content-Type-Options': 'nosniff', 'Alt-Svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000', 'Transfer-Encoding': 'chunked'},
 'artificial%20intelligence')

In [2]:
import re
data = r.json()
linkpat = re.compile("https://www\.reddit\.com/r/\w+/comments/.*")
items = [item['link'] for item in data['items'] if not linkpat.match(item['link']) is None]
items

KeyError: 'items'

In [ ]:
# get links with json
post_json_links = [link[:-1] + ".json" for link in items]
post_json_links

In [ ]:
reddit_data = [requests.get(post, headers = {'User-agent': 'your bot 0.1'}).json() for post in post_json_links]

In [ ]:
comments = []
for post in reddit_data:
    if not type(post) == list:
        print(post)
        continue
    post_data = post[1]['data']
    comments.append(post_data['children'])

comments

In [ ]:
len(comments[0])

In [ ]:
def get_body_and_replies(comment_json):
    comments_of_post = []
    for comment in comment_json:
        root_data = comment['data']
        body = root_data['body']
        comments_of_post.append(body)
        if (type(body) == dict):   
            children = root_data['replies']['data']['children']
            comments_of_post.extend([child['data']['body'] for child in children])
    return comments_of_post

In [ ]:

comments_per_post = [get_body_and_replies(comment) for comment in comments]
comments_per_post

In [ ]:
len(comments), len(items)

In [ ]:
comments_with_url = []
for i in range(len(comments_per_post)):
    for post_comment in comments_per_post[i]:
        comments_with_url.append({ "url": items[i], "comment": post_comment })

len(comments_with_url)

In [ ]:
import pandas as pd

df = pd.DataFrame(data=comments_with_url)
df

In [ ]:
!pip install emoji

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer
import emoji

nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

In [ ]:
from pandas.errors import EmptyDataError

try:
  basic_stopwords = list(
    # Handle empty data error
    pd.read_csv('basic_stopwords.txt', header=None).values.flatten()
  )
except EmptyDataError:
  basic_stopwords = []

try:
  domain_stopwords = list(
    pd.read_csv('domain_stopwords.txt', header=None).values.flatten()
  )
except EmptyDataError:
  domain_stopwords = []

def preprocess_text(corpus, text_column='text'):
  cleaned_corpus = corpus.copy()

  # Lowercase
  cleaned_corpus['cleaned_text'] = cleaned_corpus[text_column].str.lower()

  # Lemmatize (by default, lemmatize nouns)
  # Other options:
  #   'v' for verbs
  #   'a' for adjectives
  #   'r' for adverbs
  #   's' for satellites adjectives (adjectives that appear after verbs)
  lemmatizer = WordNetLemmatizer()
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].apply(
      lambda text: ' '.join(
        [lemmatizer.lemmatize(word, pos='n') for word in text.split()]
      )
  )

  # Stemmer
  stemmer = PorterStemmer()
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].apply(
      lambda text: ' '.join(
        [stemmer.stem(word) for word in text.split()]
      )
  )

  # Remove NLTK stopwords
  en_stopwords_list = stopwords.words('english')
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].apply(
    lambda text: ' '.join(
      [
        word for word in text.split() if word not in en_stopwords_list
      ]
    )
  )

  # Remove basic stopwords
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].apply(
    lambda text: ' '.join(
      [word for word in text.split() if word not in basic_stopwords]
    )
  )

  # Remove domain stopwords
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].apply(
    lambda text: ' '.join(
      [word for word in text.split() if word not in domain_stopwords]
    )
  )

  # Remove trailing and leading whitespaces
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].str.strip()

  # Remove non-alphanumeric characters
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].str.replace(r'\W', ' ', regex=True)

  # Remove numbers
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].str.replace(r'\d+', ' ', regex=True)

  # Remove emojis using emoji library
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].apply(
    lambda text: ' '.join(
      [word for word in text.split() if word not in list(emoji.EMOJI_DATA.keys())]
    )
  )

  return cleaned_corpus['cleaned_text']

In [ ]:
df['processed_text'] = preprocess_text(df, text_column='comment')
df

In [ ]:
df.to_csv("redditcomments.csv")